# Building Energy Consumption Forecasting (ASHRAE Great Energy Predictor III)

##  Project Objective
The goal of this project is to predict energy and water consumption (including water and wastewater services) for corporate and municipal facilities. 

The dataset covers approximately **2,000 buildings** across **16 global cities** over a one-year historical period, combined with corresponding meteorological data. The task is to forecast energy consumption for these facilities **1.5 years into the future**, leveraging known weather forecasts.

To solve this regression task, we implement a **Linear Regression** framework.

---

##  ETL Pipeline Overview
The Extract, Transform, Load (ETL) phase serves as the foundation of this project. It encompasses data ingestion, cleaning, and structural merging to prepare the raw data for advanced analytics, feature engineering, and model training.

For this pipeline, we extract **three primary datasets**:
1. **Building Metadata**: `building_metadata.csv.gz` — Structural features of the buildings.
2. **Target Metrics (Train)**: `train.0.0.csv.gz` — Historical energy consumption logs.
3. **Weather Data**: `weather_train.csv.gz` — Meteorological observations per location.

---

##  Data Merging and Preprocessing Strategy
The datasets are interconnected and require a multi-stage join operation before modeling:
* `train.0.0.csv.gz` links to `building_metadata.csv.gz` via the `building_id` foreign key.
* `building_metadata.csv.gz` links to `weather_train.csv.gz` via the `site_id` spatial key.

### Next Steps:
1. **Data Integration**: Perform a left join to merge all three tables into a unified dataframe based on `building_id` and `site_id` (accounting for the `timestamp` constraint).
2. **Data Cleaning**: Handle missing values (NaNs), filter out structural anomalies, and drop irrelevant observations to ensure data quality.


## Phase 1: Data Extraction and Preprocessing (ETL)
In this initial phase, we extract the compressed data from remote repositories and explore the core features.

### Features Description (Building Metadata):
* **site_id** — Anonymized ID number for the building location.
* **building_id** — Anonymized ID number for the specific building.
* **primary_use** — Primary activity indicator for the building (e.g., Education, Office).
* **square_feet** — Total gross floor area of the building.
* **year_built** — The year the building was opened.
* **floor_count** — Number of floors in the building.


In [2]:
# 1: Library Imports and Configuration
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

# Set global plotting parameters for clean visualizations
plt.rcParams['figure.figsize'] = [16, 8]
plt.style.use('seaborn-v0_8-whitegrid')  # Optional: makes plots look professional

print("Libraries imported successfully.")


Libraries imported successfully.


In [6]:
# 2: Data Ingestion (Local Extract)
from pathlib import Path
import pandas as pd

# Define the data directory path
DATA_DIR = Path("data")

print("Ingesting local datasets from the 'data' directory...")

# Load compressed local CSV files directly into Pandas DataFrames
buildings = pd.read_csv(DATA_DIR / "building_metadata.csv.gz")
weather = pd.read_csv(DATA_DIR / "weather_train.csv.gz")
energy_0 = pd.read_csv(DATA_DIR / "train.0.0.csv.gz")

print("Data extraction completed successfully.\n")

print("\n--- [1/3] Building Metadata Sample ---")
display(buildings.head(2))

print("\n--- [2/3] Weather Logs Sample ---")
display(weather.head(2))

print("\n--- [3/3] Train Logs Sample ---")
display(energy_0.head(2))

# Verify data shapes and memory footprints (Fixed Order)
for name, df in [
    ("Building Metadata", buildings),
    ("Weather Logs", weather),
    ("Train Logs (Energy)", energy_0),
]:
    print(f"-> {name}: {df.shape[0]:,} rows | {df.shape[1]} columns")


Ingesting local datasets from the 'data' directory...
Data extraction completed successfully.


--- [1/3] Building Metadata Sample ---


,site_id,building_id,primary_use,square_feet,year_built,floor_count
0,0,0,Education,7432,2008.0,NaN
1,0,1,Education,2720,2004.0,NaN



--- [2/3] Weather Logs Sample ---


,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-01-01 00:00:00,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,2016-01-01 01:00:00,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5



--- [3/3] Train Logs Sample ---


,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01 00:00:00,0.0
1,0,0,2016-01-01 01:00:00,0.0


-> Building Metadata: 1,449 rows | 6 columns
-> Weather Logs: 139,773 rows | 9 columns
-> Train Logs (Energy): 8,784 rows | 4 columns


### Data Dictionary

#### 1. Building Metadata (`buildings`)
* `site_id` — Location code (anonymized ID for the city/site).
* `building_id` — Unique identifier for each building.
* `primary_use` — Primary activity indicator (e.g., Education, Office).
* `square_feet` — Total gross floor area of the building (sq. ft.).
* `year_built` — The year the building opened.
* `floor_count` — Number of floors.

#### 2. Weather Data (`weather`)
* `site_id` — Location code matching the building metadata.
* `timestamp` — Date and time of the observation.
* `air_temperature` — Air temperature in degrees Celsius.
* `cloud_coverage` — Portion of the sky covered by clouds.
* `dew_temperature` — Dew point temperature in degrees Celsius.
* `precip_depth_1_hr` — Precipitation depth recorded in one hour (mm).
* `sea_level_pressure` — Sea-level air pressure (hPa).
* `wind_direction` — Wind direction in compass degrees (0-360).
* `wind_speed` — Wind speed in meters per second.

#### 3. Target Energy Consumption (`energy_0`)
* `building_id` — Unique identifier for each building.
* `meter` — The meter type (0 = electricity, 1 = chilledwater, 2 = steam, 3 = hotwater).
* `timestamp` — Date and time of the reading.
* `meter_reading` — Total energy consumption (target variable).


In [ ]:
# 3: Target Variable Visualization (Time Series)

# 1. Convert timestamp column from string to datetime objects
energy_0['timestamp'] = pd.to_datetime(energy_0['timestamp'])

# 2. Filter data for a specific building (e.g., building_id == 0) to avoid mixing data
# (If energy_0 already contains only building 0, you can skip this filtering step)
building_0_df = energy_0[energy_0['building_id'] == 0].set_index('timestamp')

# 3. Plot the clean time-series graph with professional styling
building_0_df['meter_reading'].plot(linewidth=1.5)
plt.title("Historical Energy Consumption Trends (Building 0)", fontsize=14, pad=15)
plt.xlabel("Timeline (Date & Time)", fontsize=12)
plt.ylabel("Energy Consumption (Meter Reading)", fontsize=12)
plt.tight_layout()
plt.show()


###  Exploratory Data Analysis (EDA) Insight
By plotting the time-series data for the target energy consumption, we can observe a critical anomaly:
* **Zero-Reading Anomaly:** From January until late May, the meter readings remain strictly at `0`. This is a known instrumentation error in the ASHRAE dataset (e.g., the meter was offline or miscalibrated).
* **Action Plan:** During the upcoming data cleaning phase, these leading zero values must be filtered out, as they will negatively skew our Linear Regression model.


In [7]:
# 4: Data Integration (ETL - Merge Phase)

print("Merging datasets into a single unified DataFrame...")

# 1. Merge core energy logs with building structural metadata
merged_df = pd.merge(energy_0, buildings, on="building_id", how="left")

# 2. Align meteorological observations using a composite key
final_df = pd.merge(
    merged_df, weather, on=["site_id", "timestamp"], how="left"
)

print(f"Data integration complete. Final shape: {final_df.shape[0]:,} rows | {final_df.shape[1]} columns")
display(final_df.head(2))


Merging datasets into a single unified DataFrame...
Data integration complete. Final shape: 8,784 rows | 16 columns


,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,2016-01-01 00:00:00,0.0,0,Education,7432,2008.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,0,2016-01-01 01:00:00,0.0,0,Education,7432,2008.0,NaN,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5


In [8]:
# 6: Missing Value Analysis

print("Analyzing missing values in the integrated dataset...")

# Calculate total number of NaN values per column
missing_count = final_df.isnull().sum()

# Filter and display only columns that have missing data, sorted by highest count
missing_summary = missing_count[missing_count > 0].sort_values(ascending=False)

print(f"Analysis complete. Found {len(missing_summary)} columns with missing data:\n")
print(missing_summary)


Analyzing missing values in the integrated dataset...
Analysis complete. Found 7 columns with missing data:

floor_count           8784
cloud_coverage        3830
wind_direction         250
sea_level_pressure      85
air_temperature          3
dew_temperature          3
precip_depth_1_hr        1
dtype: int64


### Missing Data Insights
The missing value analysis reveals 7 columns with incomplete data structures:
* **`floor_count`**: Missing exactly 8,784 records, indicating that floor data for this specific building profile is entirely absent across the full annual timeline.
* **Meteorological Features (`cloud_coverage`, `wind_direction`, `sea_level_pressure`)**: Show typical weather station sensor dropouts, with cloud coverage being the most severely affected.
* **Core Temperature Variables (`air_temperature`, `dew_temperature`)**: Contain only 3 missing values each, making them ideal candidates for local linear interpolation.


### Объединение данных в датасет building + weather + energy

In [50]:
# Объединим потребление электроэнергии и информацию о зданиях по столбцу building_id
energy_0 = pd.merge(left=energy_0, right=buildings, how="left",
                    left_on="building_id", right_on="building_id")
energy_0.head(2)

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count
0,0,0,2016-01-01 00:00:00,0.0,0,Education,7432,2008.0,NaN
1,0,0,2016-01-01 01:00:00,0.0,0,Education,7432,2008.0,NaN


In [51]:
# Объединим получившийся набор с данными по погоде. Выставим индексы для объединения timestamp, site_id
energy_0.set_index(["timestamp", "site_id"], inplace=True)
weather.set_index(["timestamp", "site_id"], inplace=True)

In [52]:
# Проведем объединение и сбросим индексы
energy_0 = pd.merge(left=energy_0, right=weather, how="left",
                    left_index=True, right_index=True)
energy_0.reset_index(inplace=True)
print(energy_0.head(2))

             timestamp  site_id  building_id  meter  meter_reading  \
0  2016-01-01 00:00:00        0            0      0            0.0   
1  2016-01-01 01:00:00        0            0      0            0.0   

  primary_use  square_feet  year_built  floor_count  air_temperature  \
0   Education         7432      2008.0          NaN             25.0   
1   Education         7432      2008.0          NaN             24.4   

   cloud_coverage  dew_temperature  precip_depth_1_hr  sea_level_pressure  \
0             6.0             20.0                NaN              1019.7   
1             NaN             21.1               -1.0              1020.2   

   wind_direction  wind_speed  
0             0.0         0.0  
1            70.0         1.5  


### Нахождение пропущенных данных

In [53]:
# Найдем пропущенные данные для дальнейшего заполнения.
# Посчитаем количество пропусков данных по столбцам
for column in energy_0.columns:
    energy_nulls = energy_0[column].isnull().sum()
    if energy_nulls > 0:
        print (column + ": " + str(energy_nulls))
print(energy_0[energy_0["precip_depth_1_hr"].isnull()])

floor_count: 8784
air_temperature: 3
cloud_coverage: 3830
dew_temperature: 3
precip_depth_1_hr: 1
sea_level_pressure: 85
wind_direction: 250
             timestamp  site_id  building_id  meter  meter_reading  \
0  2016-01-01 00:00:00        0            0      0            0.0   

  primary_use  square_feet  year_built  floor_count  air_temperature  \
0   Education         7432      2008.0          NaN             25.0   

   cloud_coverage  dew_temperature  precip_depth_1_hr  sea_level_pressure  \
0             6.0             20.0                NaN              1019.7   

   wind_direction  wind_speed  
0             0.0         0.0  


#### Алгоритм организации заполнения пропущенных данных: 

|Признак|Комментарий|
|--:|:--|
|`air_temperature`|NaN -> 0|
|`dew_temperature`|NaN -> 0|
|`cloud_coverage`|NaN -> 0|
|`precip_depth_1_hr`|NaN -> 0, -1 -> 0|
|`sea_level_pressure`|NaN -> среднее|
|`wind_direction `|NaN -> среднее (роза ветров)|

In [55]:
energy_0["air_temperature"] = energy_0["air_temperature"].fillna(0)
energy_0["cloud_coverage"].fillna(0, inplace=True)
energy_0["dew_temperature"].fillna(0, inplace=True)
# замена отрицательных осадков
energy_0["precip_depth_1_hr"] = energy_0["precip_depth_1_hr"].apply(lambda x:x if x>0 else 0) 
# Давление и ветер через среднее
energy_0_sealevel_pressure_mean = energy_0["sea_level_pressure"].mean()
energy_0["sea_level_pressure"] = energy_0["sea_level_pressure"].apply(lambda x:energy_0_sealevel_pressure_mean if x!=x else x)
energy_0_wind_direction_mean = energy_0["wind_direction"].mean()
energy_0["wind_direction"] = energy_0["wind_direction"].apply(lambda x:energy_0_wind_direction_mean if x!=x else x)
energy_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   timestamp           8784 non-null   object 
 1   site_id             8784 non-null   int64  
 2   building_id         8784 non-null   int64  
 3   meter               8784 non-null   int64  
 4   meter_reading       8784 non-null   float64
 5   primary_use         8784 non-null   object 
 6   square_feet         8784 non-null   int64  
 7   year_built          8784 non-null   float64
 8   floor_count         0 non-null      float64
 9   air_temperature     8784 non-null   float64
 10  cloud_coverage      8784 non-null   float64
 11  dew_temperature     8784 non-null   float64
 12  precip_depth_1_hr   8784 non-null   float64
 13  sea_level_pressure  8784 non-null   float64
 14  wind_direction      8784 non-null   float64
 15  wind_speed          8784 non-null   float64
dtypes: flo

/tmp/ipykernel_14456/3249371909.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  energy_0["cloud_coverage"].fillna(0, inplace=True)
/tmp/ipykernel_14456/3249371909.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [58]:
energy_0.head()

,timestamp,site_id,building_id,meter,meter_reading,primary_use,square_feet,year_built,floor_count,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,2016-01-01 00:00:00,0,0,0,0.0,Education,7432,2008.0,NaN,25.0,6.0,20.0,0.0,1019.7,0.0,0.0
1,2016-01-01 01:00:00,0,0,0,0.0,Education,7432,2008.0,NaN,24.4,0.0,21.1,0.0,1020.2,70.0,1.5
2,2016-01-01 02:00:00,0,0,0,0.0,Education,7432,2008.0,NaN,22.8,2.0,21.1,0.0,1020.2,0.0,0.0
3,2016-01-01 03:00:00,0,0,0,0.0,Education,7432,2008.0,NaN,21.1,2.0,20.6,0.0,1020.1,0.0,0.0
4,2016-01-01 04:00:00,0,0,0,0.0,Education,7432,2008.0,NaN,20.0,2.0,20.0,0.0,1020.0,250.0,2.6
